# 02 · Label Derivation

**Reads `identity_roboflow.csv` from notebook 01. Produces `labels_raw.csv`.**

## What this does
Derives a mild / moderate / severe label for every Roboflow image from colour
alone: resize, convert to CIE LAB, cluster the pixels into three tissue groups
with k-means, read off the proportions, and threshold the necrotic fraction.

## This reproduces the method under test, faithfully
The point of the project is that this derivation does not recover severity.
Notebook 03 proves that. For the proof to be honest, this notebook must run the
derivation exactly as originally specified, including its central assumption:

| cluster | luminance | assigned tissue |
|---|---|---|
| 0 | lowest L* | necrosis (dark brown to black) |
| 1 | middle L* | slough (yellow to cream) |
| 2 | highest L* | granulation (pink to red) |

Notebook 03 shows this ordering is inverted in reality — granulation sits at
L* 118, below periwound callus at L* 168 — which is one of the reasons the
labels fail. But that is a finding to be demonstrated, not an assumption to bake
in here.

## Runs on all images, copies included
The derivation labels every file, not one per photograph. Notebook 03 needs the
augmented copies to measure how often a photograph's own copies disagree, which
is one of the four validation checks.

## The recovered rule
- severe := necrosis >= 0.20
- mild := granulation >= 0.67 and slough <= 0.27
- moderate := everything else

## Two settings that are load-bearing
- `n_init = 4` in k-means. A single initialisation shifts proportions by up to
  0.166 against a 10-init reference, which is the same size as the effect being
  measured.
- clusters ordered by luminance, so a proportion column means the same thing in
  every image.

## Output
`labels_raw.csv` — identity columns plus the three proportions, the severity
label, and its integer code.


In [ ]:
# Cell 1 · config and input check
from pathlib import Path
import pandas as pd

# ── inputs, from notebook 01 ─────────────────────────────────────────
INTERIM   = Path('data/interim')
IDENTITY  = INTERIM / 'identity_roboflow.csv'
# ─────────────────────────────────────────────────────────────────────

RESIZE  = 128    # working resolution for clustering
N_INIT  = 4      # do NOT drop to 1 (see note in Cell 2)
SEED    = 42

if not IDENTITY.exists():
    print(f'STOPPING. {IDENTITY} not found.')
    print('Run 01_deduplication.ipynb first.')
    raise SystemExit(1)

ident = pd.read_csv(IDENTITY)
print(f'identity table: {len(ident):,} images, '
      f'{ident.photo_id.nunique():,} photographs')
print('deriving labels on ALL images, copies included.')
print('notebook 03 needs the copies to measure label stability.')

In [ ]:
# Cell 2 · the derivation, exactly as originally specified
#
# This REPRODUCES the method under test. Notebook 03 then shows it fails.
# The cluster-to-tissue mapping here is the ORIGINAL assumption:
#     lowest luminance  -> necrosis   (dark brown to black)
#     middle luminance  -> slough     (yellow to cream)
#     highest luminance -> granulation (pink to red)
# Notebook 03 demonstrates this ordering is inverted in reality
# (granulation L* 118 sits below periwound callus L* 168).
import numpy as np
from PIL import Image
from skimage import color
from sklearn.cluster import KMeans

def tissue_proportions(path):
    # returns (necrosis, slough, granulation) fractions summing to 1,
    # or NaNs if the image cannot be read
    try:
        with Image.open(path) as im:
            a = np.asarray(im.convert('RGB').resize((RESIZE, RESIZE)),
                           dtype=np.float32) / 255.0
    except Exception:
        return np.array([np.nan, np.nan, np.nan])
    lab = color.rgb2lab(a).reshape(-1, 3)
    # n_init stays at 4. Measured against an n_init=10 reference:
    #   n_init=4 deviates by <= 0.006, n_init=1 by up to 0.166.
    # 0.166 is the same order as the effect being measured, so the
    # cheap single-init option silently corrupts the proportions.
    km = KMeans(3, n_init=N_INIT, random_state=SEED).fit(lab)
    order = np.argsort(km.cluster_centers_[:, 0])   # ascending L*
    counts = np.bincount(km.labels_, minlength=3)[order].astype(np.float64)
    return counts / counts.sum()   # [necrosis, slough, granulation]

# quick self-test on a solid mid-grey image
print('derivation function ready (resize', RESIZE, ', n_init', N_INIT, ')')

In [ ]:
# Cell 3 · run the derivation on every image, in parallel
# Embarrassingly parallel: one image at a time, nothing shared.
# joblib spreads it across cores. Expect a few minutes for ~9,000 images.
from joblib import Parallel, delayed
import numpy as np, time, os

paths = ident.path.tolist()
n_jobs = max(1, (os.cpu_count() or 2) - 1)
print(f'clustering {len(paths):,} images on {n_jobs} cores...')

t0 = time.time()
rows = Parallel(n_jobs=n_jobs, batch_size=32, verbose=5)(
    delayed(tissue_proportions)(p) for p in paths)
props = np.vstack(rows)
print(f'done in {(time.time()-t0)/60:.1f} min')

n_bad = int(np.isnan(props).any(axis=1).sum())
if n_bad:
    print(f'  {n_bad} images failed to read (will carry NaN)')

In [ ]:
# Cell 4 · apply the threshold rules and assemble the label table
# The rule was recovered exactly from the original label file:
#     severe   := necrosis    >= 0.20
#     mild     := granulation >= 0.67 AND slough <= 0.27
#     moderate := everything else
import numpy as np

lab = ident.copy()
lab['necrosis_prop']    = props[:, 0]
lab['slough_prop']      = props[:, 1]
lab['granulation_prop'] = props[:, 2]

def grade(row):
    if np.isnan(row.necrosis_prop):
        return 'error'
    if row.necrosis_prop >= 0.20:
        return 'severe'
    if row.granulation_prop >= 0.67 and row.slough_prop <= 0.27:
        return 'mild'
    return 'moderate'

lab['severity'] = lab.apply(grade, axis=1)
lab['y'] = lab['severity'].map({'mild': 0, 'moderate': 1, 'severe': 2})

# verify the rule reproduces itself exactly (sanity, not circular:
# confirms the threshold constants match the assignment)
sev_by_rule = (lab.necrosis_prop >= 0.20)
sev_by_label = (lab.severity == 'severe')
match = (sev_by_rule == sev_by_label)[lab.severity != 'error'].mean()
print(f'severe rule reproduces the severe label on {match*100:.1f}% of rows')
print('\nclass distribution (per image):')
print(lab.severity.value_counts().to_dict())

In [ ]:
# Cell 5 · what the derivation actually depends on
# Mutual information between each channel and the final label. If severity
# collapses to a single channel, the three-channel representation is an
# illusion. This is the first quantitative hint the labels are thin.
from sklearn.metrics import mutual_info_score
import numpy as np

ok = lab.severity != 'error'
y = lab.loc[ok, 'y'].values

def mi(channel):
    # discretise the proportion into 20 bins, then MI with the label
    binned = np.digitize(lab.loc[ok, channel].values, np.linspace(0, 1, 21))
    return mutual_info_score(binned, y)

print('mutual information with the severity label (nats):')
for ch in ['necrosis_prop', 'slough_prop', 'granulation_prop']:
    print(f'  {ch:<18} {mi(ch):.3f}')

print('\ncorpus-level plausibility:')
print(f'  mean necrotic fraction : {lab.loc[ok,"necrosis_prop"].mean():.3f}')
print(f'  images above severe cut: '
      f'{(lab.loc[ok,"necrosis_prop"]>=0.20).mean()*100:.1f}%')
print('\nNo real DFU cohort is three-quarters severe. Notebook 03 shows why:')
print('the derivation grades healthy skin as severe too.')

In [ ]:
# Cell 6 · save
OUT = INTERIM / 'labels_raw.csv'
lab.to_csv(OUT, index=False)
print(f'wrote {OUT.resolve()}')
print(f'  {len(lab):,} rows: path, photo_id, patient_id, hash_cluster,')
print(f'  necrosis_prop, slough_prop, granulation_prop, severity, y')

In [ ]:
# Cell 7 · summary
print('=' * 58)
print('STAGE 02 COMPLETE')
print('=' * 58)
ok = lab.severity != 'error'
d = lab.loc[ok, 'severity'].value_counts()
for c in ['mild', 'moderate', 'severe']:
    print(f'  {c:<9} {d.get(c,0):>6,} images')
if (lab.severity == 'error').any():
    print(f'  errors  {(lab.severity=="error").sum():>6,}')
print(f'\n  labels are PER IMAGE (copies included).')
print(f'  notebook 03 validates them and consolidates to one per photograph.')
print('\nnext: 03_label_validation.ipynb')